In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# **LSTM *inter-epoch***

Este es el primer modelo secuencial del proyecto. Complementa al baseline tabular (XGBoost), que clasifica cada época de forma aislada: acá el modelo aprende explícitamente la dinámica temporal entre épocas. El sueño tiene estructura (las etapas siguen patrones tipo Wake $\to$ N1 $\to$ N2 $\to$ N3 $\to \dots \to$ REM, con transiciones que no son aleatorias), y una red recurrente puede explotar ese contexto.

El modelo es una LSTM bidireccional (BiLSTM) many-to-many: procesa una noche completa como una secuencia y emite una predicción por época. 

## **Arquitectura**
$$
\text{features}[T, F]  \to  \text{encoder}  \to  \texttt{LSTM} \ \text{(2 capas, bidireccional)}  \to  \text{Dropout} \to \text{Linear}  \to  \text{logits}[T, 5]
$$

- ***encoder***: en esta primer versión es IdentityEncoder (deja pasar las features pre-computadas tal cual). Es un punto de extensión deliberado: a futuro se reemplaza por una CNN 1D sobre la señal cruda para armar el híbrido $\texttt{CNN1D}\to\texttt{LSTM}$, sin tocar el resto del pipeline.
- ***LSTM***: $\texttt{hidden\_size=128, num\_layers=2, dropout=0.3, bidirectional=True}$. "Bidireccional" = lee la noche de adelante hacia atrás y de atrás hacia adelante, así cada época ve contexto pasado y futuro (en sleep staging offline tenemos la noche entera, no hay restricción de causalidad). El flag `bidirectional` permite pasar de LSTM a BiLSTM sin reescribir nada.
- ***Cabeza lineal***: proyecta el estado oculto de cada timestep a 5 logits, uno por etapa.
- ***Input***: una secuencia por noche, $\text{features}[T, F]$, donde $T \approx 800–900$ épocas (longitud variable por noche) y $F = 122$ features por época.
- ***Output***: $\text{logits}[T, 5]$ para cada época, un puntaje por clase. La predicción es el argmax sobre las 5 clases.
- ***Clases*** (target = *expert_label*): {0:Wake, 1:N1, 2:N2, 3:N3, 4:REM}. La etiqueta 5:Unknown no es una clase a predecir: se ignora.

## Uso de **Features Pre-computadas**

El modelo utiliza `epoch_features.csv` (ejecutar [notebooks/baseline.ipynb](baseline.ipynb)). El CSV tiene una fila por época, con columnas subject, night, epoch, <122 features>, label, dreem. El modelo:

1. Agrupa por (subject, night) $\to$ cada noche es una secuencia. Las filas se ordenan por epoch para respetar el orden temporal (`NightSequenceDataset`).
2. Usa como features todas las columnas que no son metadata (META_COLS = subject, night, epoch, label, dreem). Son las mismas features intra/inter-época que usa el baseline (HRV, acelerometría, lags/leads, medias móviles, etc.).
3. Usa label (experto) como target y conserva dreem solo como referencia (no entra al modelo).

A diferencia del baseline, que trata cada fila como un ejemplo independiente, acá las filas de una misma noche forman una sola secuencia ordenada que la LSTM recorre época por época.

## Armado y Procesamiento de **Secuencias**

**Split por sujeto**: Train/val/test se separan con sujetos disjuntos (`split_subjects`), no por noche ni por época. Esto
evita data leakage: dos noches del mismo paciente comparten fisiología, y si cayeran en train y test el modelo "haría
trampa". 

**Normalización (estandarización)**: Se calcula media y desvío de cada feature solo sobre el train (`feature_stats`,
ignorando NaN) y se aplica $(x − \mu) / \sigma$ a los tres splits. Las stats salen siempre del train para no filtrar
información de val/test.

**NaN de borde**: Las features inter-época (_lag, _lead, etc.) son NaN en los bordes de cada noche (no hay vecino). Se imputan a 0 (= la media, ya post-estandarización) con `nan_to_num`.

**Padding + packing**: Las noches tienen longitudes distintas, pero un batch necesita un tensor rectangular. En cada batch (collate_nights):
- Se paddea cada noche hasta la más larga del batch: las features con 0.0 y las labels con UNKNOWN (5).
- Se guarda un vector lengths con el largo real de cada noche.
- El modelo usa `pack_padded_sequence` / `pad_packed_sequence`: la LSTM ignora los timesteps de padding (no contaminan los estados ocultos ni los gradientes). El padding es solo un relleno para poder batchear, no datos reales.

## **Loss Function**

Se usa CrossEntropyLoss(weight=..., **ignore_index=UNKNOWN**). Tanto las épocas Unknown como el padding (ambos marcados con 5) quedan fuera de la loss, pero se mantienen dentro de la secuencia para no romper la continuidad temporal (sacarlas desalinearía el contexto que ve la recurrente).

## **Desbalance de Clases**

Las etapas están desbalanceadas (N2 abunda, N1 escasea). Se pasan pesos inversos a la frecuencia de cada clase (calculados sobre el train, `class_weights`) a la loss, para que el modelo no colapse a predecir siempre la clase mayoritaria.

## Entrenamiento

- **Optimizador**: Adam, con gradient clipping (grad_clip=5.0) para estabilizar el entrenamiento de la recurrente.
- **Métricas por época** (compute_metrics, calculadas solo sobre épocas válidas, sin padding ni Unknown): *Cohen's Kappa* (métrica principal, corrige el acuerdo por azar), *macro F1* (promedia las 5 clases por igual, robusto al desbalance), *accuracy* y *matriz de confusión*. Además se reporta una versión colapsada a 4 clases (Wake / Light=N1+N2 / Deep=N3 / REM) para comparar con el paper de referencia.
- **Checkpoint**: se guarda el mejor modelo por Kappa de validación (.pt). El archivo incluye los pesos (model_state), la
config, y las stats de normalización (mean/std). Al final, se recarga ese mejor checkpoint y se evalúa en test.
- **Reproducibilidad**: seed fija (set_seed) sobre Python/NumPy/PyTorch, y toda la config centralizada en la dataclass
Config.

In [6]:
import sys
sys.path.append("..")

In [7]:
from src.lstm import Config, train

model, history, test_metrics = train(Config())

sujetos -> train 33 | val 7 | test 7


entrenando: 100%|██████████| 60/60 [08:22<00:00,  8.37s/epoch, loss=0.2611, val_kappa=0.4146, macroF1=0.5277, acc=0.5652, best=0.4499] 



TEST (mejor ckpt, val kappa 0.4499, epoch 27):
  kappa 0.4435 | macroF1 0.5592 | acc 0.5666
  4-clases: kappa 0.4769 | macroF1 0.6448 | acc 0.6405
